# Tools and Skills · 2.1.9

A Tool performs an operation; a Skill declares capabilities and instructions. The graph defines execution. load_skill needs the linked local resources, not Studio.

In [ ]:
import pydantic
import agentic_systems as toolkit
assert toolkit.__version__ == '2.1.9'


In [ ]:
class State(pydantic.BaseModel):
    value: int

@toolkit.tool
def increment(value: int) -> dict:
    return {'value': value + 1}
graph = toolkit.Graph(State)
graph.add_node('increment', increment)
graph.add_edge(toolkit.START, 'increment')
graph.add_edge('increment', toolkit.END)
agent = toolkit.Agent('counter', graph=graph)


In [ ]:
capability = toolkit.skill(name='arithmetic', tools=(increment,), instructions='Use integer arithmetic.')
worker = toolkit.Agent('worker', graph=graph, skills=(capability,))
result = worker.run({'value': 3})
assert result.ok and result.data == {'value': 4}
print(result.human_results())
print(capability.inspect())


Portable loading: [complete folder](../../examples/portable_skills/assets/arithmetic). SKILL.md documents the capability; skill.py constructs the runtime Skill.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'examples/common_graph/application.py').is_file()))
sys.path.insert(0, str(ROOT))
loaded = toolkit.load_skill(ROOT / 'examples/portable_skills/assets/arithmetic')
loaded_result = loaded.tools[0].run({'value': 3})
assert loaded_result.ok and loaded_result.data == {'value': 4}
print(loaded.inspect())
